# 📘 Tutorial 9: Exporting Plot-ready Data

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

A clean notebook workflow should produce explicit output tables. If a later Matplotlib tutorial needs a summary table, save that table deliberately rather than requiring every plotting notebook to repeat the full cleaning pipeline.

---

## 🎯 Learning objectives

By the end of this notebook, you should be able to:

- choose what should be exported,
- use `to_csv`,
- separate raw and processed data folders,
- save clean tables for later plotting,
- describe simple reproducibility habits.


## 🔧 Setup


In [ ]:
from pathlib import Path

import pandas as pd


## 1. Locate raw and processed folders


In [ ]:
project_root = Path.cwd()
if not (project_root / "data" / "part1").exists():
    project_root = project_root.parent

data_dir = project_root / "data" / "part1"
processed_dir = data_dir / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print("Raw/example data:", data_dir)
print("Processed outputs:", processed_dir)


Raw and processed data should be separate. Raw files are inputs. Processed files are outputs that can be regenerated.


## 2. Build a plot-ready summary table


In [ ]:
measurements = pd.read_csv(data_dir / "replicate_measurements.csv")
metadata = pd.read_csv(data_dir / "sample_metadata.csv")

joined = measurements.drop(columns=["condition"]).merge(
    metadata,
    on="sample_id",
    how="left",
    validate="many_to_one",
)

joined.head()


In [ ]:
plot_ready = (
    joined.groupby(["condition", "concentration_mM"], as_index=False)
    .agg(
        mean_response=("response", "mean"),
        sd_response=("response", "std"),
        n=("response", "count"),
    )
    .sort_values("concentration_mM")
)

plot_ready["sem_response"] = plot_ready["sd_response"] / (plot_ready["n"] ** 0.5)
plot_ready


This table is small but complete enough for a later error-bar plot.


## 3. Choose export columns deliberately


In [ ]:
export_table = plot_ready[
    ["condition", "concentration_mM", "mean_response", "sem_response", "n"]
].copy()

export_table


Export only the columns that are useful for the next workflow. Keep raw data and richer intermediate tables in the notebook or as separate outputs if needed.


## 4. Export with `to_csv`


In [ ]:
output_path = processed_dir / "summary_for_plotting.csv"
export_table.to_csv(output_path, index=False)

print(output_path)
print("Saved:", output_path.exists())


`index=False` avoids saving the pandas row index as an extra unnamed CSV column.


## 5. Read the exported file back


In [ ]:
reloaded = pd.read_csv(output_path)
reloaded


In [ ]:
assert list(reloaded.columns) == list(export_table.columns)
assert len(reloaded) == len(export_table)


Reading the exported file back is a simple sanity check. It confirms that the table a later notebook will read has the expected shape and columns.


## 6. Reproducibility habits

Good export habits include:

- keep raw and processed data separate,
- make the cleaning steps rerunnable,
- use descriptive filenames,
- avoid saving accidental index columns,
- read exported files back during development,
- document what each exported table represents.


## 🧭 Key takeaways

- Export plot-ready tables when they are useful downstream.
- `to_csv(..., index=False)` is the usual choice for clean tabular outputs.
- Processed files should be reproducible from raw inputs and notebook code.
- Exported tables should have clear column names and no hidden assumptions.
- Later plotting notebooks should be able to focus on figures, not data repair.
